Importation du bibliothèque ensuite création de u codeblock les données pour YOLO.

In [ ]:
from pathlib import Path

def convert_bbox_to_yolo_format(x1, y1, x2, y2, img_width, img_height): 
    x_center = ((x1 + x2) / 2) / 8--
    y_center = ((y1 + y2) / 2) / img_height
    width = (x2 - x1) / img_width
    height = (y2 - y1) / img_height
    return x_center, y_center, width, height

base_url = Path(r"C:\Users\Mohamed Walid\Desktop\Internship\Code\Subset_LE")

# join paths using /
img_path = base_url / "Image" / "1.jpg"
loc_path = base_url / "groundtruth_localization" / "1.txt"

import cv2
img = cv2.imread(str(img_path))
h, w = img.shape[:2]


with open(loc_path) as f:
    x1, y1, x2, y2 = map(float, f.read().split())

yolo_bbox = convert_bbox_to_yolo_format(x1, y1, x2, y2, w, h)


output_path = base_url / "groundtruth_recognition" / "1.txt"

with open(output_path, "w") as f:
    f.write(f"0 {yolo_bbox[0]} {yolo_bbox[1]} {yolo_bbox[2]} {yolo_bbox[3]}")

In [5]:
print(img_path)

C:\Users\Mohamed Walid\Desktop\Internship\Code\Subset_LE\Image\1.jpg


Préparation de la base des données

In [ ]:
from pathlib import Path
import cv2
import random
import shutil

# === BASE PATH (Code folder) ===
base = Path(r"C:\Users\Mohamed Walid \Desktop\Internship\Code")

# Input folders
images_src = base / "Image"
labels_src = base / "Subset_LE" / "groundtruth_localization"

# Output dataset
dataset = base / "Dataset"

# Create folders
for p in [
    dataset / "images/train",
    dataset / "images/val",
    dataset / "labels/train",
    dataset / "labels/val",
]:
    p.mkdir(parents=True, exist_ok=True)

# === FUNCTION ===
def convert_bbox_to_yolo_format(x1, y1, x2, y2, w, h):
    x_center = ((x1 + x2) / 2) / w
    y_center = ((y1 + y2) / 2) / h
    width = (x2 - x1) / w
    height = (y2 - y1) / h
    return x_center, y_center, width, height

# === LOAD IMAGES ===
image_files = list(images_src.glob("*.jpg"))
random.shuffle(image_files)

# Split 80/20
split_idx = int(0.8 * len(image_files))
train_files = image_files[:split_idx]
val_files = image_files[split_idx:]

# === PROCESS FUNCTION ===
def process(files, split):
    for img_path in files:
        name = img_path.stem
        loc_path = labels_src / f"{name}.txt"

        # Read image
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"❌ Cannot read {img_path}")
            continue

        h, w = img.shape[:2]

        # Read bbox
        try:
            with open(loc_path) as f:
                x1, y1, x2, y2 = map(float, f.read().split())
        except:
            print(f"❌ Missing or invalid bbox for {name}")
            continue

        # Convert
        x_center, y_center, bw, bh = convert_bbox_to_yolo_format(x1, y1, x2, y2, w, h)

        # Copy image
        dst_img = dataset / f"images/{split}/{img_path.name}"
        shutil.copy(img_path, dst_img)

        # Save label
        label_path = dataset / f"labels/{split}/{name}.txt"
        with open(label_path, "w") as f:
            f.write(f"0 {x_center} {y_center} {bw} {bh}")

# Run
process(train_files, "train")
process(val_files, 
        "val")

print("✅ Dataset successfully created!")

Model training

In [1]:
from ultralytics import YOLO

# yolov8n.pt = small model to start
model = YOLO("yolov8n.pt")

# Train
model.train(data="dataset.yaml", epochs=50)

New https://pypi.org/project/ultralytics/8.4.36 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.10.20 torch-2.11.0+cpu CPU (Intel Core i5-8250U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train9, nbs=64, nms=False, op

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001321AD3FF10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [4]:
from ultralytics import YOLO

In [5]:
model = YOLO("C:\\Users\\Mohamed Walid\\Desktop\\Internship\\Code\\runs\\detect\\train9\\weights\\best.pt")

In [6]:
from pathlib import Path

runs_path = Path(r"C:/Users/Mohamed Walid/Desktop/Internship/Code/runs/detect")
print(list(runs_path.glob("*/weights/*.pt")))

[WindowsPath('C:/Users/Mohamed Walid/Desktop/Internship/Code/runs/detect/train8/weights/best.pt'), WindowsPath('C:/Users/Mohamed Walid/Desktop/Internship/Code/runs/detect/train8/weights/last.pt'), WindowsPath('C:/Users/Mohamed Walid/Desktop/Internship/Code/runs/detect/train9/weights/best.pt'), WindowsPath('C:/Users/Mohamed Walid/Desktop/Internship/Code/runs/detect/train9/weights/last.pt')]


In [ ]:
import cv2
from pathlib import Path

# Pick a few images
images_path = Path("./Dataset/images/train")
labels_path = Path("./Dataset/labels/train")

image_files = list(images_path.glob("*.jpg"))[:10]  # first 10 images

for img_path in image_files:
    image = cv2.imread(str(img_path))
    h, w = image.shape[:2]

    label_path = labels_path / f"{img_path.stem}.txt"

    if not label_path.exists():
        print(f"❌ Missing label: {img_path.stem}")
        continue

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        cls, xc, yc, bw, bh = map(float, line.strip().split())

        # Convert YOLO → pixel coords
        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)
        x2 = int((xc + bw/2) * w)
        y2 = int((yc + bh/2) * h)

        # Draw rectangle
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

    cv2.imshow("Check", image)
    cv2.waitKey(0)

cv2.destroyAllWindows()

In [ ]:
results = model(img_path, show=True)

In [ ]:
base_url = Path(r"C:\Users\Mohamed Walid\Desktop\Internship\Code\Subset_LE")

# join paths using /
img_path = base_url / "Image" / "2.jpg"

In [ ]:
import easyocr
import cv2

# Load image
img = cv2.imread(r"C:\Users\Mohamed Walid\Desktop\Internship\Code\Subset_LE/image/1.jpg")

# Create OCR reader
reader = easyocr.Reader(['en'])

# Read text
results = reader.readtext(img)

# Print detected text
for r in results:
    print(r[1])